# KVQuant Implementation -- 2-bit only

One of four split-out notebooks (2-bit / 3-bit / 4-bit / full-precision
baseline), each running independently so they can execute in parallel
Colab sessions instead of one long combined run. This notebook is
identical in methodology to `KVQuant_Implementation_v3.ipynb`, scoped to
just the 2-bit quantized model -- no other bit-width, no baseline model,
loaded or evaluated here (the baseline lives in its own dedicated
notebook, `KVQuant_Baseline_Implementation.ipynb`, so it only needs to run
once rather than being redundantly repeated in all three quantized
notebooks).

Every fix already built into v3 is preserved as-is: real hook-based KV
cache memory measurement (via `register_outlier_trackers`, using the
actual `get_outliers`/`get_outliers_dynamic` functions and the model's own
real calibrated thresholds -- not an assumed fixed sparsity fraction),
teacher-forced step-by-step TTFT/TBT timing for WikiText-103, real
`generate()`-based GSM8K timing with TBT excluding TTFT, H2O-matching deterministic random sampling (up to 256 non-overlapping
WikiText-103 chunks and up to 1,024 valid examples from GSM8K,
ARC-Challenge, and HellaSwag, all with seed 42), and
download-retry-with-cache-clear robustness for every network call.

Run cells top to bottom. Needs a GPU runtime.

## Setup

In [ ]:
!hostname

In [ ]:
# Block 1 - Environment setup
# Run once per fresh runtime. Package versions are pinned so environment
# differences are never a confound between compression methods.

import os
os.environ["HF_HUB_DISABLE_XET"] = "1"

from google.colab import drive
drive.mount("/content/drive")

!python -m pip install -q --no-deps \
  "transformers==4.43.4" \
  "accelerate==0.33.0" \
  "tokenizers==0.19.1" \
  "huggingface_hub==0.36.2" \
  sentencepiece \
  einops

!python -m pip install -q \
  "datasets==2.14.5" \
  tqdm \
  matplotlib

!python -m pip install -q --no-deps --force-reinstall "huggingface_hub==0.36.2"

import os

os.environ["HF_HUB_DISABLE_XET"] = "1"

try:
    from google.colab import userdata
    _hf_token = userdata.get("HF_TOKEN")
except Exception:
    _hf_token = os.environ.get("HF_TOKEN")

if _hf_token:
    from huggingface_hub import login
    login(token=_hf_token)
    print("Logged in to HuggingFace")
else:
    print("No HF_TOKEN found -- Llama-3.1-8B is GATED: this will fail to load without a token that has accepted the Meta license at https://huggingface.co/meta-llama/Llama-3.1-8B")

print("Block 1 finished. Now run Block 2.")

In [ ]:
# Block 2 - Imports, GPU check

import gc
import math
import re
import time
import random
import pickle
import sys

import numpy as np
import torch
import torch.nn as nn
import pandas as pd

import datasets
import transformers
import huggingface_hub
from datasets import load_dataset
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, StoppingCriteria, StoppingCriteriaList

print("numpy:", np.__version__)
print("pandas:", pd.__version__)
print("torch:", torch.__version__)
print("cuda:", torch.version.cuda)
print("datasets:", datasets.__version__)
print("transformers:", transformers.__version__)
print("huggingface_hub:", huggingface_hub.__version__)
print("gpu:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO CUDA")

HAS_CUDA = torch.cuda.is_available()
DEVICE = torch.device("cuda" if HAS_CUDA else "cpu")
MODEL_DTYPE = torch.bfloat16 if HAS_CUDA else torch.float32


def clear_hf_dataset_cache(*dataset_names):
    """Removes cached files for the given HF dataset repo name(s) (e.g.
    "wikitext", "gsm8k") from both the datasets cache and the hub cache.
    Used as an on_retry hook: a download that breaks partway through can
    leave a corrupted partial file that every subsequent retry just
    resumes (and re-breaks at the same point) instead of truly restarting
    -- clearing the cache forces a genuinely fresh download."""
    home = os.path.expanduser("~")
    for name in dataset_names:
        for base in [
            os.path.join(home, ".cache", "huggingface", "datasets", name),
            os.path.join(home, ".cache", "huggingface", "hub", f"datasets--{name}"),
        ]:
            shutil.rmtree(base, ignore_errors=True)


def robust_call(fn, *args, max_retries=5, backoff_sec=5, desc="operation", on_retry=None, **kwargs):
    """Retries fn(*args, **kwargs) on any exception, up to max_retries times,
    waiting backoff_sec between attempts -- guards dataset downloads against
    transient network failures (e.g. IncompleteRead/ChunkedEncodingError)
    rather than letting one flaky connection kill the whole notebook run.
    If on_retry is given, it's called (no args) after each failure, before
    the next attempt -- e.g. clear_hf_dataset_cache, so a retry that hit a
    stuck/corrupted partial download actually starts fresh instead of
    resuming (and re-breaking at) the same point every time."""
    last_err = None
    for attempt in range(1, max_retries + 1):
        try:
            return fn(*args, **kwargs)
        except Exception as e:
            last_err = e
            _msg = f"  {desc}: attempt {attempt}/{max_retries} failed ({e!r})"
            if attempt < max_retries:
                _msg += f", retrying in {backoff_sec}s..."
            print(_msg)
            if attempt < max_retries:
                if on_retry is not None:
                    on_retry()
                time.sleep(backoff_sec)
    raise last_err


if not HAS_CUDA:
    print("WARNING: No GPU detected. This will be very slow.")

# NOTE: clear_hf_dataset_cache/robust_call are defined here (not in Helper
# Functions, below) because the Fisher calibration cell later in this same
# Setup section calls robust_call for its wikitext-2 prefetch -- they need
# to exist before that point. sync_if_cuda/clear_memory have no such early
# dependency and live in Helper Functions with the rest of the genuinely
# cross-dataset inference machinery.

In [ ]:
# Block 3 - Experiment settings.
# Shared sampling policy used by every H2O and KVQuant comparison notebook:
#   - WikiText-103: up to 256 random non-overlapping chunks
#   - GSM8K, ARC-Challenge, HellaSwag: up to 1,024 random valid examples
#   - deterministic seed: 42
# Sampling happens after chunk construction or validity filtering. Selected
# indices are sorted back into source order so the subset is random while the
# evaluation order remains stable across methods.

LOCAL_MODEL_PATH = "/content/llama-3.1-8b"
HF_MODEL_ID = "meta-llama/Llama-3.1-8B"
MODEL_ID = LOCAL_MODEL_PATH if __import__("os").path.exists(LOCAL_MODEL_PATH) else HF_MODEL_ID

SHARED_SEED = 42
C = 2048
WIKITEXT_EVAL_SAMPLES = 256
QA_EVAL_SAMPLES = 1024

# Backward-compatible aliases used by a few downstream cells/comments.
N_EVAL_SAMPLES = WIKITEXT_EVAL_SAMPLES
N_QA_SAMPLES = QA_EVAL_SAMPLES
GSM8K_MAX_NEW_TOKENS = 256
METHOD_NAME = "kvquant_2bit"

ABITS = 2                   # this notebook's single bit-width
SPARSITY_THRESHOLD = 0.99   # keep the extreme ~1% of values in full precision
FISHER_OUTPUT_DIR = "/content/fisher-llama-3.1-8b-c2048"
QUANTIZER_PATH = "/content/quantizers_llama-3.1-8b_v3_c2048_abits2.pickle"

random.seed(SHARED_SEED)
np.random.seed(SHARED_SEED)
torch.manual_seed(SHARED_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SHARED_SEED)


def seeded_subset(items, max_samples, seed=SHARED_SEED):
    """Select a reproducible random subset, then restore source order.

    A fresh RNG is created on every call, so running notebook sections in a
    different order cannot change which examples are selected.
    """
    items = list(items)
    sample_count = min(int(max_samples), len(items))
    selected_indices = sorted(
        random.Random(int(seed)).sample(range(len(items)), sample_count)
    )
    return [items[index] for index in selected_indices], selected_indices

GSM8K_FEWSHOT_PREFIX = (
    "You are solving grade-school math word problems.\n"
    "Show the calculation step by step, then end with exactly this format:\n"
    "#### <final number>\n\n"

    "Question: There are 15 trees in the grove. Grove workers will plant trees today. After they are done, there will be 21 trees. How many trees did the grove workers plant today?\n"
    "Answer: There are 15 trees originally. After planting, there are 21 trees. So the workers planted 21 - 15 = 6 trees.\n"
    "#### 6\n\n"

    "Question: If there are 3 cars in the parking lot and 2 more cars arrive, how many cars are in the parking lot?\n"
    "Answer: There are originally 3 cars. 2 more cars arrive. 3 + 2 = 5 cars.\n"
    "#### 5\n\n"

    "Question: Leah had 32 chocolates and her sister had 42. If they ate 35, how many pieces do they have left in total?\n"
    "Answer: Leah and her sister started with 32 + 42 = 74 chocolates. After eating 35, they have 74 - 35 = 39 left.\n"
    "#### 39\n\n"

    "Question: Jason had 20 lollipops. He gave Denny some lollipops. Now Jason has 12 lollipops. How many lollipops did Jason give to Denny?\n"
    "Answer: Jason started with 20 lollipops and now has 12. So he gave away 20 - 12 = 8 lollipops.\n"
    "#### 8\n\n"

    "Question: Shawn has five toys. For Christmas, he got two toys each from his mom and dad. How many toys does he have now?\n"
    "Answer: Shawn started with 5 toys. He got 2 from mom and 2 from dad, which is 2 + 2 = 4 more toys. 5 + 4 = 9 toys total.\n"
    "#### 9\n\n"

    "Question: There were nine computers in the server room. Five more computers were installed each day, from Monday to Thursday. How many computers are now in the server room?\n"
    "Answer: 4 days from Monday to Thursday, with 5 computers installed each day, is 4 * 5 = 20 computers added. 9 + 20 = 29 computers total.\n"
    "#### 29\n\n"

    "Question: Michael had 58 golf balls. On Tuesday, he lost 23 golf balls. On Wednesday, he lost 2 more. How many golf balls did he have at the end of Wednesday?\n"
    "Answer: Michael started with 58 golf balls. After losing 23 on Tuesday, he had 58 - 23 = 35. After losing 2 more on Wednesday, he had 35 - 2 = 33 golf balls.\n"
    "#### 33\n\n"

    "Question: Olivia has $23. She bought five bagels for $3 each. How much money does she have left?\n"
    "Answer: Five bagels at $3 each cost 5 * 3 = 15 dollars. Olivia started with $23, so she has 23 - 15 = 8 dollars left.\n"
    "#### 8\n"
)

print("Model:", MODEL_ID)
print("Method:", METHOD_NAME)
print("Chunk length C:", C)
print("Bit-width:", ABITS, "| sparsity threshold:", SPARSITY_THRESHOLD)
print("Random sampling seed:", SHARED_SEED)
print("WikiText-103 random chunk target:", WIKITEXT_EVAL_SAMPLES)
print("GSM8K/ARC/HellaSwag random example target:", QA_EVAL_SAMPLES)
print("GSM8K max new tokens:", GSM8K_MAX_NEW_TOKENS)

In [ ]:
# Repo setup 1/3 - Clone repo (always fresh, so the rotary_emb patch below
# always applies to pristine upstream source rather than a possibly-
# already-patched copy left over from an earlier run in this session).
import os, shutil


def check_path(path, label):
    if not os.path.exists(path):
        raise FileNotFoundError(f"ERROR: {label} not found at {path}")
    print(f"OK: {label} found at {path}")


if os.path.exists("/content/KVCacheCompression"):
    shutil.rmtree("/content/KVCacheCompression")
    print("Removed existing repo copy for a clean re-clone")

!git clone --recurse-submodules https://github.com/yoshikodes/KVCacheCompression.git

assert os.path.exists("/content/KVCacheCompression/KVQuant/gradients/run-fisher.py"), \
    "ERROR: run-fisher.py not found! Clone may have failed."
assert os.path.exists("/content/KVCacheCompression/KVQuant/quant/llama_simquant.py"), \
    "ERROR: llama_simquant.py not found! Clone may have failed."
print("Repo verified OK")

In [ ]:
# Repo setup 2/3 - Install gradients (Fisher calibration) dependencies
check_path("/content/KVCacheCompression/KVQuant/gradients", "gradients dir")

%cd /content/KVCacheCompression/KVQuant/gradients
!pip install -e . -q
!pip install datasets==2.14.5 sentencepiece==0.1.99 accelerate -q -U
!pip install peft==0.6.0 -q
print("Gradients deps installed")

In [ ]:
# Patch - llama_simquant.py: move the shared rotary embedding module to the
# target device before calibration. transformers==4.43.4 computes rotary
# embeddings ONCE at the top-level LlamaModel, not per-layer -- the upstream
# script's custom per-layer CPU->GPU transfer (written for an older
# architecture) never moves this shared module, so it stays on CPU while
# everything it's multiplied against is on GPU. Only needed for the
# calibration path here (this notebook's own evaluation, below, uses a
# normally-loaded model and never calls llama_simquant.py's llama_eval).
filepath = "/content/KVCacheCompression/KVQuant/quant/llama_simquant.py"
with open(filepath, "r") as f:
    _content = f.read()

if "model.model.rotary_emb = model.model.rotary_emb.to(DEV)" not in _content:
    _m = re.search(r"^([ \t]*)quantizers = llama_calibration\(", _content, re.MULTILINE)
    assert _m, "ERROR: could not locate the llama_calibration(...) call to patch"
    _indent = _m.group(1)
    _insertion = (
        f"{_indent}if hasattr(model, 'model') and hasattr(model.model, 'rotary_emb'):\n"
        f"{_indent}    model.model.rotary_emb = model.model.rotary_emb.to(DEV)\n"
    )
    _content = _content[:_m.start()] + _insertion + _content[_m.start():]
    with open(filepath, "w") as f:
        f.write(_content)
    print("Patched llama_simquant.py: rotary_emb moved to device before calibration")
else:
    print("rotary_emb device patch already applied, skipping")

In [ ]:
# Patch - llama_simquant.py: get_model() hardcodes
# use_flash_attention_2=True, which crashes with "flash_attn seems to be not
# installed" since this project never installs the flash-attn package (it's
# optional elsewhere and skipped here entirely). Replace it with a
# GPU-capability-aware attn_implementation choice that also respects a
# FORCE_ATTN_IMPL env var, matching KVQuant_Implementation_v2.ipynb's Patch
# 2.4. Without this, both the Quantize step and this notebook's own
# in-process model loading would need it -- but the in-process loader
# already builds its own model_kwargs directly (no use_flash_attention_2
# anywhere in this notebook's own code), so only llama_simquant.py's
# internal get_model() needs patching, for the Quantize step.
filepath = "/content/KVCacheCompression/KVQuant/quant/llama_simquant.py"
with open(filepath, "r") as f:
    _content = f.read()

_old_load = "model = AutoModelForCausalLM.from_pretrained(model, config=config, trust_remote_code=True, use_flash_attention_2=True, torch_dtype=torch.half)"
_new_load = (
    "import warnings as _warnings\n"
    "    import os as _attn_os\n"
    "    import torch as _torch_gpu_check\n"
    "    _cap = _torch_gpu_check.cuda.get_device_capability(0) if _torch_gpu_check.cuda.is_available() else (0, 0)\n"
    "    _attn_impl = \"flash_attention_2\" if _cap[0] >= 8 else \"sdpa\"\n"
    "    _force_attn = _attn_os.environ.get('FORCE_ATTN_IMPL')\n"
    "    if _force_attn:\n"
    "        _attn_impl = _force_attn\n"
    "    print(f'GPU compute capability: {_cap} -> using attn_implementation={_attn_impl}' + (' (forced)' if _force_attn else ''))\n"
    "    with _warnings.catch_warnings():\n"
    "        _warnings.filterwarnings(\"ignore\", message=\".*Flash Attention 2.0 with a model not initialized on GPU.*\")\n"
    "        model = AutoModelForCausalLM.from_pretrained(model, config=config, trust_remote_code=True, attn_implementation=_attn_impl, torch_dtype=torch.half)"
)
if _old_load in _content:
    _content = _content.replace(_old_load, _new_load)
    with open(filepath, "w") as f:
        f.write(_content)
    print("llama_simquant.py attn_implementation patch applied (GPU-capability-aware)")
elif "GPU compute capability" in _content:
    print("attn_implementation patch already applied, skipping")
else:
    print("WARNING: could not find the expected model-loading line to patch.")
print("Has GPU-capability check:", "GPU compute capability" in open(filepath).read())

In [ ]:
# Patch - simquant_module_quantizer.py: QuantLinearSim.__init__ builds
# self.outlier_threshold_upper/lower via torch.tensor(already_a_tensor),
# which PyTorch flags with a UserWarning ("recommended to use
# sourceTensor.detach().clone()..."). Purely cosmetic -- quantizer[0]/[1]
# are already tensors (SimQuant.quantize() builds them that way before
# pickling), so this produces a numerically identical copy either way --
# but silenced here since it's a one-line-per-side change and this file
# gets read fresh from a clean clone every run.
filepath = "/content/KVCacheCompression/KVQuant/quant/kvquant/simquant_module_quantizer.py"
with open(filepath, "r") as f:
    _content = f.read()

_old_thresh = (
    "        self.outlier_threshold_upper = torch.tensor(quantizer[0]).cuda().flatten().half()\n"
    "        self.outlier_threshold_lower = torch.tensor(quantizer[1]).cuda().flatten().half()"
)
_new_thresh = (
    "        self.outlier_threshold_upper = quantizer[0].clone().detach().cuda().flatten().half()\n"
    "        self.outlier_threshold_lower = quantizer[1].clone().detach().cuda().flatten().half()"
)
if _old_thresh in _content:
    _content = _content.replace(_old_thresh, _new_thresh)
    with open(filepath, "w") as f:
        f.write(_content)
    print("Patched simquant_module_quantizer.py: outlier thresholds built via .clone().detach() instead of torch.tensor(tensor)")
elif "quantizer[0].clone().detach()" in _content:
    print("simquant_module_quantizer.py outlier-threshold patch already applied, skipping")
else:
    print("WARNING: could not find the expected outlier-threshold lines to patch (cosmetic only -- safe to ignore if this shows up).")

In [ ]:
# Patch - simquant_module_quantizer.py: QuantLinearSim.forward() hardcodes
# `x = x.half()` before the quantized-layer matmul and `y = y.half()` on
# the way out, unconditionally forcing fp16 regardless of what dtype the
# model actually runs in. This was a silent no-op as long as MODEL_DTYPE
# was float16 (self.weight, captured from the real k_proj/v_proj weight in
# __init__, was already fp16 too), but MODEL_DTYPE is now bfloat16 -- x
# arrives bf16, gets force-cast to fp16, then `x @ self.weight` fails with
# "expected mat1 and mat2 to have the same dtype, but got: Half !=
# BFloat16" since self.weight was never cast and is still bf16. Fixed by
# keying both casts off self.weight.dtype (== MODEL_DTYPE, whatever it is)
# instead of a hardcoded fp16 -- this also avoids silently handing bf16
# attention layers an fp16 K/V tensor back out of forward(), which would
# just move the same dtype mismatch one level up into the attention matmul.
filepath = "/content/KVCacheCompression/KVQuant/quant/kvquant/simquant_module_quantizer.py"
with open(filepath, "r") as f:
    _content = f.read()

_old_input_cast = "        x = x.half() # for now cast to fp16 and back (quantization code assumes fp32)\n        y = x @ self.weight"
_new_input_cast = "        x = x.to(self.weight.dtype)  # match the model's real dtype (fp16 or bf16), not a hardcoded fp16\n        y = x @ self.weight"

_old_output_cast = "        y = y.half()\n        return y"
_new_output_cast = "        y = y.to(self.weight.dtype)\n        return y"

if _old_input_cast in _content:
    _content = _content.replace(_old_input_cast, _new_input_cast)
    _content = _content.replace(_old_output_cast, _new_output_cast)
    with open(filepath, "w") as f:
        f.write(_content)
    print("Patched simquant_module_quantizer.py: QuantLinearSim.forward() now casts to self.weight.dtype instead of hardcoded fp16")
elif "x = x.to(self.weight.dtype)" in _content:
    print("simquant_module_quantizer.py dtype patch already applied, skipping")
else:
    raise RuntimeError("ERROR: could not locate the hardcoded .half() cast to patch in simquant_module_quantizer.py")


In [ ]:
# Patch - gradients/src/transformers: add rope_type="llama3" support to the
# vendored transformers fork run-fisher.py imports (via PYTHONPATH below).
# This fork predates Llama 3.1's NTK-by-parts long-context RoPE scaling --
# its _rope_scaling_validation only accepts the old {"type", "factor"}
# 2-key format, and its rotary embedding classes only implement "linear"/
# "dynamic". Llama-3.1-8B's real config.json rope_scaling is
# {"factor": 8.0, "low_freq_factor": 1.0, "high_freq_factor": 4.0,
# "original_max_position_embeddings": 8192, "rope_type": "llama3"}, which
# the unpatched fork rejects outright (ValueError from AutoConfig, raised
# inside the run-fisher.py subprocess). Patched to match HF's own
# ROPE_INIT_FUNCTIONS["llama3"] formula exactly, so Fisher calibration
# sees the SAME positional encoding the model actually uses -- the
# frequency scaling applies uniformly to every position, not just beyond
# the original 8192 context, so this isn't a cosmetic no-op even at the
# short C=2048 calibration length used here.
config_path = "/content/KVCacheCompression/KVQuant/gradients/src/transformers/models/llama/configuration_llama.py"
modeling_path = "/content/KVCacheCompression/KVQuant/gradients/src/transformers/models/llama/modeling_llama.py"

with open(config_path, "r") as f:
    _cfg_content = f.read()

_old_validation = """        if not isinstance(self.rope_scaling, dict) or len(self.rope_scaling) != 2:
            raise ValueError(
                "`rope_scaling` must be a dictionary with with two fields, `type` and `factor`, "
                f"got {self.rope_scaling}"
            )
        rope_scaling_type = self.rope_scaling.get("type", None)
        rope_scaling_factor = self.rope_scaling.get("factor", None)
        if rope_scaling_type is None or rope_scaling_type not in ["linear", "dynamic"]:
            raise ValueError(
                f"`rope_scaling`'s type field must be one of ['linear', 'dynamic'], got {rope_scaling_type}"
            )
        if rope_scaling_factor is None or not isinstance(rope_scaling_factor, float) or rope_scaling_factor <= 1.0:
            raise ValueError(f"`rope_scaling`'s factor field must be a float > 1, got {rope_scaling_factor}")"""

_new_validation = """        rope_scaling_type = self.rope_scaling.get("type", self.rope_scaling.get("rope_type", None))
        if rope_scaling_type == "llama3":
            required = {"factor", "low_freq_factor", "high_freq_factor", "original_max_position_embeddings"}
            missing = required - set(self.rope_scaling)
            if missing:
                raise ValueError(f"`rope_scaling` of type 'llama3' is missing required field(s): {missing}")
            return
        if not isinstance(self.rope_scaling, dict) or len(self.rope_scaling) != 2:
            raise ValueError(
                "`rope_scaling` must be a dictionary with with two fields, `type` and `factor`, "
                f"got {self.rope_scaling}"
            )
        rope_scaling_factor = self.rope_scaling.get("factor", None)
        if rope_scaling_type is None or rope_scaling_type not in ["linear", "dynamic"]:
            raise ValueError(
                f"`rope_scaling`'s type field must be one of ['linear', 'dynamic'], got {rope_scaling_type}"
            )
        if rope_scaling_factor is None or not isinstance(rope_scaling_factor, float) or rope_scaling_factor <= 1.0:
            raise ValueError(f"`rope_scaling`'s factor field must be a float > 1, got {rope_scaling_factor}")"""

if _old_validation in _cfg_content:
    _cfg_content = _cfg_content.replace(_old_validation, _new_validation)
    with open(config_path, "w") as f:
        f.write(_cfg_content)
    print("Patched configuration_llama.py: rope_scaling validation now accepts rope_type=\'llama3\'")
elif "llama3" in _cfg_content:
    print("configuration_llama.py rope_scaling patch already applied, skipping")
else:
    raise RuntimeError("ERROR: could not locate rope_scaling validation block to patch in configuration_llama.py")

with open(modeling_path, "r") as f:
    _model_content = f.read()

_llama3_class = '''

class LlamaLlama3ScalingRotaryEmbedding(LlamaRotaryEmbedding):
    """LlamaRotaryEmbedding extended with Llama 3\'s NTK-by-parts long-context
    scaling (rope_scaling rope_type="llama3"). Reimplements HF\'s
    ROPE_INIT_FUNCTIONS["llama3"] formula exactly, since this vendored fork
    predates that scaling type."""

    def __init__(self, dim, max_position_embeddings=2048, base=10000, device=None,
                 scaling_factor=8.0, low_freq_factor=1.0, high_freq_factor=4.0,
                 original_max_position_embeddings=8192):
        self.scaling_factor = scaling_factor
        self.low_freq_factor = low_freq_factor
        self.high_freq_factor = high_freq_factor
        self.original_max_position_embeddings = original_max_position_embeddings
        super().__init__(dim, max_position_embeddings, base, device)

    def _set_cos_sin_cache(self, seq_len, device, dtype):
        self.max_seq_len_cached = seq_len

        base_inv_freq = 1.0 / (self.base ** (torch.arange(0, self.dim, 2, dtype=torch.int64).float().to(device) / self.dim))

        low_freq_wavelen = self.original_max_position_embeddings / self.low_freq_factor
        high_freq_wavelen = self.original_max_position_embeddings / self.high_freq_factor
        wavelen = 2 * math.pi / base_inv_freq

        inv_freq_llama = torch.where(wavelen > low_freq_wavelen, base_inv_freq / self.scaling_factor, base_inv_freq)
        smooth_factor = (
            (self.original_max_position_embeddings / wavelen - self.low_freq_factor)
            / (self.high_freq_factor - self.low_freq_factor)
        )
        smoothed_inv_freq = smooth_factor * inv_freq_llama / self.scaling_factor + (1 - smooth_factor) * inv_freq_llama
        is_medium_freq = ~(wavelen < high_freq_wavelen) * ~(wavelen > low_freq_wavelen)
        inv_freq_llama = torch.where(is_medium_freq, smoothed_inv_freq, inv_freq_llama)

        self.register_buffer("inv_freq", inv_freq_llama, persistent=False)

        t = torch.arange(self.max_seq_len_cached, device=device, dtype=torch.int64).type_as(self.inv_freq)
        freqs = torch.outer(t, self.inv_freq)
        emb = torch.cat((freqs, freqs), dim=-1)
        self.register_buffer("cos_cached", emb.cos().to(dtype), persistent=False)
        self.register_buffer("sin_cached", emb.sin().to(dtype), persistent=False)
'''

_class_anchor = "def rotate_half(x):"
if "class LlamaLlama3ScalingRotaryEmbedding" not in _model_content:
    _idx = _model_content.index(_class_anchor)
    _model_content = _model_content[:_idx] + _llama3_class.lstrip("\n") + "\n\n" + _model_content[_idx:]
    with open(modeling_path, "w") as f:
        f.write(_model_content)
    print("Patched modeling_llama.py: added LlamaLlama3ScalingRotaryEmbedding class")
else:
    print("modeling_llama.py LlamaLlama3ScalingRotaryEmbedding already added, skipping")

with open(modeling_path, "r") as f:
    _model_content = f.read()

_old_init_rope = '''        else:
            scaling_type = self.config.rope_scaling["type"]
            scaling_factor = self.config.rope_scaling["factor"]
            if scaling_type == "linear":'''
_new_init_rope = '''        else:
            scaling_type = self.config.rope_scaling.get("type", self.config.rope_scaling.get("rope_type"))
            scaling_factor = self.config.rope_scaling["factor"]
            if scaling_type == "llama3":
                self.rotary_emb = LlamaLlama3ScalingRotaryEmbedding(
                    self.head_dim,
                    max_position_embeddings=self.max_position_embeddings,
                    scaling_factor=scaling_factor,
                    low_freq_factor=self.config.rope_scaling["low_freq_factor"],
                    high_freq_factor=self.config.rope_scaling["high_freq_factor"],
                    original_max_position_embeddings=self.config.rope_scaling["original_max_position_embeddings"],
                    base=self.rope_theta,
                    device="cpu"
                )
            elif scaling_type == "linear":'''

if _old_init_rope in _model_content:
    _model_content = _model_content.replace(_old_init_rope, _new_init_rope)
    with open(modeling_path, "w") as f:
        f.write(_model_content)
    print("Patched modeling_llama.py: _init_rope now handles rope_type=\'llama3\'")
elif 'scaling_type == "llama3"' in _model_content:
    print("modeling_llama.py _init_rope patch already applied, skipping")
else:
    raise RuntimeError("ERROR: could not locate _init_rope's scaling_type branch to patch in modeling_llama.py")


In [ ]:
# Fisher calibration (gradients) -- seqlen/maxseqlen MUST match the Quantize
# step's seqlen below (C = 2048), or Fisher's saved tensors won't align with
# Quantize's activation tensor shapes elementwise.
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Pre-fetch wikitext-2 (with retries) in this notebook's own Python process
# before launching the Fisher subprocess -- run-fisher.py internally calls
# load_dataset('wikitext', 'wikitext-2-raw-v1', ...) itself, and a flaky
# connection there (IncompleteRead/ChunkedEncodingError) can crash the
# whole subprocess uncaught, leaving FISHER_OUTPUT_DIR never created.
print("Pre-fetching wikitext-2 (train+test) with retries...")
robust_call(
    load_dataset, "wikitext", "wikitext-2-raw-v1", split="train",
    desc="wikitext-2 train prefetch", on_retry=lambda: clear_hf_dataset_cache("wikitext"),
)
robust_call(
    load_dataset, "wikitext", "wikitext-2-raw-v1", split="test",
    desc="wikitext-2 test prefetch", on_retry=lambda: clear_hf_dataset_cache("wikitext"),
)
print("wikitext-2 cached.")


# Pre-fetch the Llama-3.1-8B model weights (with retries), using
# huggingface_hub.snapshot_download directly -- NOT transformers'
# AutoModelForCausalLM. By this point in the notebook, Cell 5's gradients
# install has replaced the real `transformers` package on disk with its
# own vendored fork (intentional -- run-fisher.py's subprocess needs it),
# but this kernel already imported the REAL transformers back in Cell 2.
# Calling AutoModelForCausalLM.from_pretrained() here would use those
# stale cached (real-transformers) references against fork files now on
# disk -- a mismatch that throws ModuleNotFoundError. snapshot_download
# only touches huggingface_hub, which isn't affected by any of this, so
# it's safe to call in-process at this point in the notebook.
from huggingface_hub import snapshot_download

print(f"Pre-fetching {MODEL_ID} weights with retries (this can take a few minutes)...")
robust_call(
    snapshot_download, MODEL_ID,
    desc=f"{MODEL_ID} weights prefetch",
    max_retries=5, backoff_sec=10,
    on_retry=lambda: clear_hf_dataset_cache(),  # no-op placeholder; see note below
)
print(f"{MODEL_ID} weights cached.")

%cd /content/KVCacheCompression/KVQuant/gradients

!CUDA_VISIBLE_DEVICES=0 PYTHONPATH=/content/KVCacheCompression/KVQuant/gradients/src python run-fisher.py --model_name_or_path {MODEL_ID} --output_dir {FISHER_OUTPUT_DIR} --dataset wikitext2 --seqlen {C} --maxseqlen {C} --num_examples 8

check_path(FISHER_OUTPUT_DIR, "Fisher output")
print("Fisher done for Llama-3.1-8B at seqlen", C)

In [ ]:
# Repo setup 3/3 - Install quant (eval-support) dependencies. Moved to run
# AFTER Fisher calibration (matching KVQuant_Implementation_v2.ipynb's
# proven cell ordering) since gradients needs tokenizers<0.19 while this
# pins tokenizers==0.19.1 -- installing this first would break Fisher.
check_path("/content/KVCacheCompression/KVQuant/quant", "quant dir")

%cd /content/KVCacheCompression/KVQuant/quant
!pip install -e . -q --no-deps

!pip install -q --no-deps \
  "transformers==4.43.4" \
  "accelerate==0.33.0" \
  "tokenizers==0.19.1" \
  "huggingface_hub==0.36.2"

print("Quant deps installed, package versions pinned to match every other method's notebook")

In [ ]:
# Quantize (nuq2, 1% outliers) -- builds the single quantizer pickle this
# notebook's own in-process eval loop will load later. Note this only runs
# llama_calibration internally (the --quantize branch), never llama_eval --
# so the rotary_emb patch above is the only one needed.
sys.path = [p for p in sys.path if "gradients" not in p]

check_path(FISHER_OUTPUT_DIR, "Fisher output for Llama-3.1-8B")

%cd /content/KVCacheCompression/KVQuant/quant

!CUDA_VISIBLE_DEVICES=0 PYTHONPATH=/content/KVCacheCompression/KVQuant/quant FORCE_ATTN_IMPL=sdpa python llama_simquant.py {MODEL_ID} --abits {ABITS} --nsamples 8 --seqlen {C} --nuq --fisher {FISHER_OUTPUT_DIR} --quantize --include_sparse --sparsity-threshold {SPARSITY_THRESHOLD} --quantizer-path {QUANTIZER_PATH}

check_path(QUANTIZER_PATH, "Quantizer pickle for Llama-3.1-8B")
print("Quantization done for Llama-3.1-8B at seqlen", C, "bit-width", ABITS)

In [ ]:
# Block - Load tokenizer + the single 2-bit quantized model (via
# make_quant_sim applied in-process, matching llama_simquant.py's own
# main-flow layer-name matching: Keys quantized per-channel pre-RoPE,
# Values quantized per-token). No full-precision baseline is loaded here
# -- that model lives entirely in KVQuant_Baseline_Implementation.ipynb.
#
# Also registers forward hooks on every quantized k_proj/v_proj
# (QuantLinearSim) submodule that recompute the REAL outlier mask the
# model actually uses internally -- via the exact same get_outliers /
# get_outliers_dynamic functions and the SAME stored threshold tensors the
# layer itself uses -- so memory can later be measured from the real,
# data-dependent outlier counts rather than an assumed fixed fraction.
# These hooks only OBSERVE (read the layer's real input, recompute its
# projection output independently); they never modify what the model
# actually computes or caches.
#
# The hooks stay attached for the model's whole lifetime (forward hooks
# fire on EVERY call to that module, including the TIMED step-by-step loop
# and the TIMED generate() call, not just the dedicated untimed memory
# passes) -- so _tracker_enabled gates the hook's actual work: it's a
# no-op unless explicitly turned on around the untimed measurement passes
# (see measure_chunk_kv_memory / generate_gsm8k_kvquant below). Without
# this gate, every timed forward call would redundantly pay the hook's
# extra x @ weight recompute plus the full outlier-mask computation,
# inflating TTFT/TBT/latency with overhead the full-precision baseline
# never pays (memory/perplexity/accuracy are unaffected either way, since
# the hook never touches the model's actual output, and
# reset_outlier_tracker already discards any stray accumulation before
# each real measurement -- this gate only removes wasted work, not a
# correctness bug in the reported numbers themselves).
%cd /content/KVCacheCompression/KVQuant/quant
sys.path.insert(0, "/content/KVCacheCompression/KVQuant/quant")

from kvquant.simquant_module_quantizer import make_quant_sim, get_outliers, get_outliers_dynamic, QuantLinearSim

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=False, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

_model_kwargs = {
    "torch_dtype": MODEL_DTYPE,
    "low_cpu_mem_usage": True,
    "attn_implementation": "sdpa",
    "trust_remote_code": True,
}
if HAS_CUDA:
    _model_kwargs["device_map"] = {"": 0}

print(f"Loading model to be quantized at {ABITS}-bit...")
model_q = AutoModelForCausalLM.from_pretrained(MODEL_ID, **_model_kwargs)
if not HAS_CUDA:
    model_q = model_q.to(DEVICE)
model_q.eval()
model_q.config.use_cache = True

device = next(model_q.parameters()).device

with open(QUANTIZER_PATH, "rb") as f:
    quantizers = pickle.load(f)

PERCHANNEL_MATCH = ["k_proj"]
PERTOKEN_MATCH = ["v_proj"]

perchannelquant = {}
pertokenquant = {}
for _k in quantizers.keys():
    for _p in PERCHANNEL_MATCH:
        if _p in _k:
            perchannelquant[_k] = quantizers[_k]
    for _p in PERTOKEN_MATCH:
        if _p in _k:
            pertokenquant[_k] = quantizers[_k]

print(f"Quantizer covers {len(perchannelquant)} per-channel (K) layers, {len(pertokenquant)} per-token (V) layers")

make_quant_sim(
    model_q, perchannelquant, ABITS,
    perchannel=True, include_sparse=True, sparsity_threshold=SPARSITY_THRESHOLD,
    dynamicquantization=False, nuq=True, nf_nuq=False, norm=False,
    cap_outliers=-1, first_few_fp16=-1, clamp=False,
)
make_quant_sim(
    model_q, pertokenquant, ABITS,
    perchannel=False, include_sparse=True, sparsity_threshold=SPARSITY_THRESHOLD,
    dynamicquantization=True, nuq=True, nf_nuq=False, norm=False,
    cap_outliers=-1, first_few_fp16=-1, clamp=False,
)

_tracker_enabled = [False]  # gate for the hooks below -- see comment above


def register_outlier_trackers(model_obj):
    """Attaches a forward hook to every QuantLinearSim submodule that
    independently recomputes that layer's projection output (x @ weight +
    bias -- identical to QuantLinearSim.forward()'s own math, computed
    BEFORE any RoPE/reshaping) and re-derives the REAL outlier mask via the
    same get_outliers/get_outliers_dynamic functions and stored threshold
    tensors the layer itself uses. Returns (tracker, handles) -- tracker
    accumulates real n_outliers/n_total per layer; call
    reset_outlier_tracker(tracker) before each measurement pass. The hook
    only does its work while _tracker_enabled[0] is True (see module-level
    note above) -- callers must toggle that flag on/off around the
    specific untimed forward pass they want measured.
    """
    tracker = {}
    handles = []

    for _name, _module in model_obj.named_modules():
        if not isinstance(_module, QuantLinearSim):
            continue
        tracker[_name] = {"n_outliers": 0, "n_total": 0}

        def _make_hook(mod, key):
            def _hook(_mod, inputs, _output):
                if not _tracker_enabled[0]:
                    return
                x = inputs[0].reshape(-1, inputs[0].shape[-1]).to(mod.weight.dtype)  # match the model's real dtype, not a hardcoded fp16 -- see the matching QuantLinearSim.forward() patch above
                weight = mod.weight.to(x.device)
                y = x @ weight
                if mod.bias is not None:
                    y = y + mod.bias.to(x.device)
                y = y.float()

                if mod.include_sparse:
                    if mod.dynamicquantization:
                        mask = get_outliers_dynamic(
                            y, channel=mod.ochannel, thresh=mod.sparsity_threshold,
                            first_few_fp16=mod.first_few_fp16,
                        )
                    else:
                        mask = get_outliers(
                            y, channel=mod.ochannel,
                            outlier_threshold_upper=mod.outlier_threshold_upper,
                            outlier_threshold_lower=mod.outlier_threshold_lower,
                            cap_outliers=mod.cap_outliers,
                            first_few_fp16=mod.first_few_fp16,
                        )
                    tracker[key]["n_outliers"] += int(mask.sum().item())
                tracker[key]["n_total"] += int(y.numel())
            return _hook

        handles.append(_module.register_forward_hook(_make_hook(_module, _name)))

    return tracker, handles


def reset_outlier_tracker(tracker):
    for _counts in tracker.values():
        _counts["n_outliers"] = 0
        _counts["n_total"] = 0


def measure_bytes_from_tracker(tracker, bits_per_element):
    total_bytes = 0
    for _counts in tracker.values():
        n_total = _counts["n_total"]
        n_outliers = _counts["n_outliers"]
        n_regular = n_total - n_outliers
        total_bytes += math.ceil(n_regular * bits_per_element / 8.0)
        total_bytes += n_outliers * 2  # fp16 value per outlier
        total_bytes += math.ceil(n_total / 8.0)  # 1-bit-per-element bitmap marking which positions are outliers
    return total_bytes


outlier_tracker, _outlier_hook_handles = register_outlier_trackers(model_q)
print(f"Quantization applied to model_q's k_proj/v_proj layers in place; {len(outlier_tracker)} outlier-tracking hooks registered (gated off by default).")
print("model_q: nuq", ABITS, "bit, sparsity_threshold", SPARSITY_THRESHOLD)

## Helper Functions

Shared inference machinery used across all three datasets (WikiText-103,
GSM8K, ARC-Challenge).

In [ ]:
# Block - sync_if_cuda/clear_memory: used across every timed inference
# loop in this notebook (WikiText-103, GSM8K, ARC-Challenge) for
# timing-safe GPU synchronization and between-dataset memory cleanup.


def sync_if_cuda():
    if HAS_CUDA:
        torch.cuda.synchronize()


def clear_memory():
    gc.collect()
    if HAS_CUDA:
        torch.cuda.empty_cache()

In [ ]:
# Block - KV-cache memory measurement (measured from REAL outlier-hook
# data) -- shared inference machinery used by both the WikiText-103 driver
# (via score_chunk_kvquant, WikiText-103 section) and directly by
# ARC-Challenge's gold-choice scoring (ARC-Challenge section). GSM8K
# measures its own memory inline in generate_gsm8k_kvquant instead of
# calling this, but this is still genuine cross-dataset machinery, not
# something specific to any one dataset's driver logic.
#
# Always run as a SEPARATE, UNTIMED pass -- callers run it AFTER their own
# timed loop finishes, so it cannot affect any timing number -- and
# _tracker_enabled is only flipped on for the duration of this one untimed
# forward call, so the outlier-tracking hooks stay inert (immediate no-op)
# everywhere else.
#
# Memory comes from register_outlier_trackers' hooks (model-loading cell,
# Setup): resetting the tracker, running one untimed forward pass over the
# whole chunk with tracking enabled, then reading the REAL per-layer
# outlier counts those hooks captured -- not an assumed fixed sparsity
# fraction.


@torch.no_grad()
def measure_chunk_kv_memory(model_obj, chunk_ids_1d, bits_per_element, tracker=None):
    chunk_ids = chunk_ids_1d.unsqueeze(0).to(device)

    if tracker is not None:
        reset_outlier_tracker(tracker)
        _tracker_enabled[0] = True
        try:
            model_obj(input_ids=chunk_ids, use_cache=False, return_dict=True)
        finally:
            _tracker_enabled[0] = False
        return measure_bytes_from_tracker(tracker, bits_per_element)

    outputs = model_obj(input_ids=chunk_ids, use_cache=True, return_dict=True)
    pkv = outputs.past_key_values
    legacy = pkv.to_legacy_cache() if hasattr(pkv, "to_legacy_cache") else pkv
    total_bytes = 0
    for layer_kv in legacy:
        for t in layer_kv:
            total_bytes += t.numel() * t.element_size()
    return total_bytes

In [ ]:
# Block - Shared multiple-choice (MC) scoring machinery, used by BOTH
# ARC-Challenge and HellaSwag (the two likelihood-based MC datasets in
# this notebook) -- genuinely dataset-agnostic: each dataset's own section
# only builds (prompt, choices, gold_index) and calls
# score_mc_question_kvquant; everything else lives here.
#
# lm_eval_encode_pair(context, choice)
#     Tokenizes context+continuation JOINTLY then splits the result at the
#     context's own token count (matching lm-evaluation-harness) --
#     preserves true tokenizer boundary behavior, which separately
#     tokenizing context and continuation can miss (BPE merges can span
#     the boundary). Left-truncates the context if the combined request
#     would exceed the model's window C.
#
# score_mc_choice_kvquant(model_obj, prompt, choice, bits_per_element, tracker)
#     Times the FULL step-by-step walk over one answer choice's tokens
#     (teacher-forced, exactly like scoring a WikiText-103 chunk) --
#     EVERY choice for a question gets this same real, timed treatment,
#     not just the correct one, since answering a multiple-choice
#     question genuinely requires scoring every candidate (there's no
#     single "real generation" the way GSM8K has). Memory is measured via
#     measure_chunk_kv_memory (previous cell) over the full processed
#     sequence.
#
# score_mc_question_kvquant(model_obj, prompt, choices, gold_index, bits_per_element, tracker)
#     Scores every choice via score_mc_choice_kvquant, then reports BOTH
#     standard MC accuracy metrics: raw (argmin of un-normalized NLL) and
#     normalized (argmin of NLL divided by the choice's own raw character
#     length -- the standard lm-evaluation-harness acc_norm convention).
#     Perplexity comes ONLY from the gold choice's own mean NLL (mirroring
#     GSM8K/ARC's "score only the gold answer" convention for perplexity).
#     TTFT = mean across choices; TBT = weighted mean across every
#     choice's decode steps (not a mean-of-per-choice-TBTs, since choices
#     can have very different lengths); total latency = SUM across
#     choices (the real cost of answering the question is the cost of
#     scoring every choice); peak memory = MAX across choices.


def lm_eval_encode_pair(context, choice):
    context = str(context)
    continuation = " " + str(choice)

    n_spaces = len(context) - len(context.rstrip())
    if n_spaces > 0:
        continuation = context[-n_spaces:] + continuation
        context = context[:-n_spaces]

    if not context:
        raise ValueError("MC context cannot be empty.")

    whole_ids = tokenizer(context + continuation, add_special_tokens=True)["input_ids"]
    context_ids = tokenizer(context, add_special_tokens=True)["input_ids"]
    continuation_ids = whole_ids[len(context_ids):]

    if not context_ids:
        raise ValueError("Context tokenization produced no tokens.")
    if not continuation_ids:
        raise ValueError(f"Continuation tokenization produced no tokens. Context={context!r}, choice={choice!r}")
    if len(continuation_ids) > int(C):
        raise ValueError(f"Choice has {len(continuation_ids)} tokens, exceeding C={C}.")

    max_context_tokens = int(C) + 1 - len(continuation_ids)
    if max_context_tokens < 1:
        raise ValueError("No room remains for a context token.")
    context_ids = context_ids[-max_context_tokens:]

    return context_ids, continuation_ids


@torch.no_grad()
def score_mc_choice_kvquant(model_obj, prompt, choice, bits_per_element, tracker=None):
    context_ids, continuation_ids = lm_eval_encode_pair(prompt, choice)
    full_ids_1d = torch.tensor(context_ids + continuation_ids, device=device)
    n_context = len(context_ids)

    chunk_ids = full_ids_1d.unsqueeze(0)
    input_ids = chunk_ids[:, :-1]
    target_ids = chunk_ids[:, 1:]

    loss_fct = nn.CrossEntropyLoss()
    nll_sum = 0.0
    scored = 0
    step_times = []
    past_key_values = None

    for pos in range(input_ids.shape[1]):
        step_input = input_ids[:, pos:pos + 1]

        sync_if_cuda()
        t0 = time.perf_counter()
        outputs = model_obj(
            input_ids=step_input, past_key_values=past_key_values,
            use_cache=True, return_dict=True,
        )
        sync_if_cuda()
        step_times.append(time.perf_counter() - t0)

        past_key_values = outputs.past_key_values
        step_logits = outputs.logits[:, -1, :]
        step_target = target_ids[:, pos]

        if pos + 1 >= n_context:
            loss = loss_fct(step_logits, step_target)
            nll_sum += loss.float().item()
            scored += 1

    ttft_sec = step_times[0]
    decode_time_sum = sum(step_times[1:])
    decode_steps = len(step_times) - 1
    total_latency_sec = sum(step_times)

    peak_bytes = measure_chunk_kv_memory(model_obj, full_ids_1d, bits_per_element, tracker)

    return {
        "nll_sum": nll_sum, "scored": scored,
        "ttft_sec": ttft_sec, "decode_time_sum": decode_time_sum, "decode_steps": decode_steps,
        "total_latency_sec": total_latency_sec, "peak_memory_bytes": peak_bytes,
        "choice_char_len": max(len(str(choice)), 1),
    }


@torch.no_grad()
def score_mc_question_kvquant(model_obj, prompt, choices, gold_index, bits_per_element, tracker=None):
    choice_results = [
        score_mc_choice_kvquant(model_obj, prompt, choice, bits_per_element, tracker)
        for choice in choices
    ]

    normalized_nlls = [r["nll_sum"] / r["choice_char_len"] for r in choice_results]

    raw_prediction = int(min(range(len(choice_results)), key=lambda i: choice_results[i]["nll_sum"]))
    normalized_prediction = int(min(range(len(choice_results)), key=lambda i: normalized_nlls[i]))

    gold_result = choice_results[gold_index]
    gold_mean_nll = gold_result["nll_sum"] / max(gold_result["scored"], 1)

    total_decode_time = sum(r["decode_time_sum"] for r in choice_results)
    total_decode_steps = sum(r["decode_steps"] for r in choice_results)

    return {
        "raw_prediction": raw_prediction,
        "normalized_prediction": normalized_prediction,
        "raw_correct": int(raw_prediction == gold_index),
        "normalized_correct": int(normalized_prediction == gold_index),
        "perplexity": math.exp(min(gold_mean_nll, 50.0)),
        "ttft_sec": sum(r["ttft_sec"] for r in choice_results) / len(choice_results),
        "tbt_sec": (total_decode_time / total_decode_steps) if total_decode_steps > 0 else 0.0,
        "total_latency_sec": sum(r["total_latency_sec"] for r in choice_results),
        "peak_memory_bytes": max(r["peak_memory_bytes"] for r in choice_results),
    }

## WikiText-103

In [ ]:
# Block - WikiText-103 random sampling.
# Load the FULL test text, concatenate it into one token stream, split it into
# non-overlapping chunks of length C, then randomly select up to 256 chunks
# with seed 42. The random subset is identical across methods. Selected chunk
# indices are sorted into source order before evaluation so timing is not
# affected by an arbitrary shuffled execution order.


def chunk_sequence(token_ids_1d, chunk_len):
    chunks = []
    n = token_ids_1d.shape[0]
    for start in range(0, n, chunk_len):
        chunks.append(token_ids_1d[start:start + chunk_len])
    return chunks


def load_wikitext103_chunks():
    testdata = robust_call(
        load_dataset, "wikitext", "wikitext-103-raw-v1", split="test",
        desc="WikiText-103 test load", on_retry=lambda: clear_hf_dataset_cache("wikitext"),
    )
    texts = [text for text in testdata["text"] if len(text.strip()) > 0]
    full_text = "\n\n".join(texts)
    ids = tokenizer(full_text, return_tensors="pt")["input_ids"][0]
    all_chunks = chunk_sequence(ids, C)
    selected_chunks, selected_indices = seeded_subset(
        all_chunks,
        WIKITEXT_EVAL_SAMPLES,
        SHARED_SEED,
    )

    print(
        f"WikiText-103 test set: {len(texts)} non-empty lines, "
        f"{ids.shape[0]} tokens, {len(all_chunks)} available non-overlapping chunks"
    )
    print(
        f"WikiText-103 selected: {len(selected_chunks)} random chunks "
        f"(requested {WIKITEXT_EVAL_SAMPLES}, seed={SHARED_SEED})"
    )
    print("WikiText-103 selected chunk indices (first 20):", selected_indices[:20])
    return selected_chunks


wikitext103_chunks = load_wikitext103_chunks()


In [ ]:
# Block - WikiText-103 step-by-step eval function. TTFT/TBT/perplexity/
# accuracy come ONLY from the timed step-by-step loop below -- the model
# is stepped one token at a time using its own KV cache, teacher-forced
# (real corpus tokens fed in, never the model's own prediction). Memory is
# measured via measure_chunk_kv_memory (Helper Functions, Setup) in a
# SEPARATE, UNTIMED pass run AFTER the timed loop finishes, so it cannot
# affect any timing number.


@torch.no_grad()
def score_chunk_kvquant(model_obj, chunk_ids_1d, bits_per_element, tracker=None):
    L = chunk_ids_1d.shape[0]
    if L < 2:
        return None
    chunk_ids = chunk_ids_1d.unsqueeze(0).to(device)
    input_ids = chunk_ids[:, :-1]
    target_ids = chunk_ids[:, 1:]

    loss_fct = nn.CrossEntropyLoss()
    nll_sum = 0.0
    correct = 0
    scored = 0
    step_times = []
    past_key_values = None

    for pos in range(input_ids.shape[1]):
        step_input = input_ids[:, pos:pos + 1]

        sync_if_cuda()
        t0 = time.perf_counter()
        outputs = model_obj(
            input_ids=step_input, past_key_values=past_key_values,
            use_cache=True, return_dict=True,
        )
        sync_if_cuda()
        step_times.append(time.perf_counter() - t0)

        past_key_values = outputs.past_key_values
        step_logits = outputs.logits[:, -1, :]
        step_target = target_ids[:, pos]

        loss = loss_fct(step_logits, step_target)
        nll_sum += loss.float().item()

        pred = step_logits.argmax(dim=-1)
        correct += int((pred == step_target).sum().item())
        scored += 1

    ttft_sec = step_times[0]
    tbt_sec = sum(step_times[1:]) / len(step_times[1:]) if len(step_times) > 1 else 0.0
    total_latency_sec = sum(step_times)

    peak_bytes = measure_chunk_kv_memory(model_obj, chunk_ids_1d, bits_per_element, tracker)

    return {
        "nll_sum": nll_sum, "scored": scored, "correct": correct,
        "ttft_sec": ttft_sec, "tbt_sec": tbt_sec,
        "total_latency_sec": total_latency_sec, "peak_memory_bytes": peak_bytes,
    }


@torch.no_grad()
def preview_chunk_prediction(model_obj, chunk_ids_1d, n_preview_tokens=30):
    """Untimed, separate forward pass over just the first n_preview_tokens of
    a chunk -- for printing what the model actually predicted vs. the real
    next tokens. Does not affect any reported metric."""
    n_preview_tokens = min(n_preview_tokens, chunk_ids_1d.shape[0] - 1)
    if n_preview_tokens < 1:
        return None
    preview_ids = chunk_ids_1d[:n_preview_tokens + 1].unsqueeze(0).to(device)
    input_ids = preview_ids[:, :-1]
    target_ids = preview_ids[:, 1:]

    outputs = model_obj(input_ids=input_ids, use_cache=False, return_dict=True)
    preds = outputs.logits.argmax(dim=-1)

    return {
        "prompt_text": tokenizer.decode(input_ids[0], skip_special_tokens=True),
        "actual_next_text": tokenizer.decode(target_ids[0], skip_special_tokens=True),
        "predicted_next_text": tokenizer.decode(preds[0], skip_special_tokens=True),
    }

In [ ]:
# Block - WikiText-103 driver: run every chunk through
# score_chunk_kvquant against the 2-bit quantized model.


def evaluate_lm_dataset_kvquant(dataset_name, chunks, model_obj, bits_per_element, method_label, tracker=None):
    clear_memory()
    total_nll = 0.0
    total_scored = 0
    total_correct = 0
    ttft_values, tbt_values, latency_values, peak_mem_values = [], [], [], []
    per_chunk_records = []

    N_PREVIEW_CHUNKS = 5
    for chunk_idx, chunk_ids in enumerate(tqdm(chunks, desc=f"{dataset_name} | {method_label}")):
        if chunk_idx < N_PREVIEW_CHUNKS:
            preview = preview_chunk_prediction(model_obj, chunk_ids)
            if preview is not None:
                print(f"\n--- {dataset_name} | {method_label} | chunk {chunk_idx} preview ---")
                print(f"Prompt:              {preview['prompt_text']!r}")
                print(f"Actual next text:    {preview['actual_next_text']!r}")
                print(f"Predicted next text: {preview['predicted_next_text']!r}")

        result = score_chunk_kvquant(model_obj, chunk_ids, bits_per_element, tracker)
        if result is None:
            continue
        total_nll += result["nll_sum"]
        total_scored += result["scored"]
        total_correct += result["correct"]
        ttft_values.append(result["ttft_sec"])
        tbt_values.append(result["tbt_sec"])
        latency_values.append(result["total_latency_sec"])
        peak_mem_values.append(result["peak_memory_bytes"])

        chunk_scored = result["scored"]
        per_chunk_records.append({
            "chunk_index": chunk_idx,
            "prompt": tokenizer.decode(chunk_ids, skip_special_tokens=True),
            "accuracy": (result["correct"] / chunk_scored) if chunk_scored > 0 else float("nan"),
            "perplexity": math.exp(min(result["nll_sum"] / chunk_scored, 50.0)) if chunk_scored > 0 else float("nan"),
            "ttft_sec": result["ttft_sec"],
            "tbt_sec": result["tbt_sec"],
            "total_latency_sec": result["total_latency_sec"],
            "peak_memory_gb": result["peak_memory_bytes"] / 1024**3,
        })

    avg_nll = total_nll / max(total_scored, 1)
    perplexity = math.exp(min(avg_nll, 50.0))
    accuracy = total_correct / max(total_scored, 1)

    os.makedirs("/content/drive/MyDrive/KVQuant_v3_Results", exist_ok=True)
    _per_prompt_path = f"/content/drive/MyDrive/KVQuant_v3_Results/{method_label}_wikitext103_per_prompt.csv"
    pd.DataFrame(per_chunk_records).to_csv(_per_prompt_path, index=False)
    print(f"Saved {len(per_chunk_records)} per-chunk {dataset_name} rows to {_per_prompt_path}")

    return {
        "dataset": dataset_name,
        "method": method_label,
        "perplexity": perplexity,
        "accuracy": accuracy,
        "ttft_sec": sum(ttft_values) / len(ttft_values) if ttft_values else float("nan"),
        "tbt_sec": sum(tbt_values) / len(tbt_values) if tbt_values else float("nan"),
        "avg_total_latency_sec": sum(latency_values) / len(latency_values) if latency_values else float("nan"),
        "peak_memory_gb": max(peak_mem_values) / 1024**3 if peak_mem_values else 0.0,
        "average_memory_gb": (sum(peak_mem_values) / len(peak_mem_values) / 1024**3) if peak_mem_values else 0.0,
    }


lm_results = []
for _name, _chunks in [("WikiText-103", wikitext103_chunks)]:
    lm_results.append(evaluate_lm_dataset_kvquant(_name, _chunks, model_q, ABITS, METHOD_NAME, tracker=outlier_tracker))

lm_results_df = pd.DataFrame(lm_results)
display(lm_results_df)

## GSM8K

In [ ]:
# Block - GSM8K loading: canonical test split, then random sampling.
# Build the complete list of valid question/answer pairs first, then select up
# to 1,024 examples with seed 42. This guarantees every method sees the same
# reproducible random questions rather than the first examples in the split.


def extract_gsm8k_gold_answer(answer_text):
    match = re.search(r"####\s*(-?[0-9][0-9,]*\.?[0-9]*)", answer_text)
    if not match:
        return None
    try:
        return float(match.group(1).replace(",", ""))
    except ValueError:
        return None


gsm8k_test = robust_call(
    load_dataset, "gsm8k", "main", split="test",
    desc="GSM8K test load", on_retry=lambda: clear_hf_dataset_cache("gsm8k"),
)

all_gsm8k_pairs = []
for item in gsm8k_test:
    gold = extract_gsm8k_gold_answer(item["answer"])
    if gold is not None:
        all_gsm8k_pairs.append({
            "question": item["question"],
            "gold": gold,
            "gold_text": item["answer"],
        })

gsm8k_qa_pairs, gsm8k_selected_indices = seeded_subset(
    all_gsm8k_pairs,
    QA_EVAL_SAMPLES,
    SHARED_SEED,
)

print(
    f"GSM8K: {len(all_gsm8k_pairs)} valid questions available; "
    f"selected {len(gsm8k_qa_pairs)} random questions "
    f"(requested {QA_EVAL_SAMPLES}, seed={SHARED_SEED})"
)
print("GSM8K selected valid-item indices (first 20):", gsm8k_selected_indices[:20])


In [ ]:
# Block - GSM8K generation with KVQuant. Uses the real model.generate()
# call -- prefill processes the whole prompt in one shot (building the
# quantized KV cache automatically, since model_q's k_proj/v_proj were
# already swapped for quantizing layers), then decode continues one token
# at a time exactly like a normal generate() loop. A StoppingCriteria
# timestamps every generated token so TTFT/TBT come from this SAME call
# that produces the graded answer -- not a separate probe. Memory is
# measured in a SEPARATE, untimed pass after generate() returns, so it
# cannot affect any timing number -- _tracker_enabled is only flipped on
# for that one untimed pass, so the outlier-tracking hooks stay inert
# during the entire timed generate() call above.
#
# Perplexity is measured on the model's OWN generated answer, live from
# this same call -- no separate teacher-forced pass. generate() is called
# with output_scores=True, return_dict_in_generate=True, which returns the
# per-step logits it already computes internally (no extra forward pass)
# alongside the token ids. After generate() returns (i.e. fully outside
# the timed window), we scan the generated tokens for four consecutive
# '#' characters (the "####" marker GSM8K answers use), then accumulate
# the log-probability of each token the model chose for itself, starting
# right after the marker and stopping the moment another '#' appears. If
# the model never generates "####" at all, perplexity is None for that
# question.


def _extract_final_number(text):
    m = re.search(r"####\s*(-?[0-9][0-9,]*\.?[0-9]*)", text)
    if m:
        num_str = m.group(1)
    else:
        nums = re.findall(r"-?[0-9][0-9,]*\.?[0-9]*", text)
        if not nums:
            return None
        num_str = nums[-1]
    num_str = num_str.replace(",", "").rstrip(".")
    try:
        return float(num_str)
    except ValueError:
        return None


class _TimingCriteria(StoppingCriteria):
    def __init__(self):
        self.token_times = []

    def __call__(self, input_ids, scores, **kwargs):
        self.token_times.append(time.perf_counter())
        return False


@torch.no_grad()
def generate_gsm8k_kvquant(model_obj, question, bits_per_element, tracker=None):
    prompt = GSM8K_FEWSHOT_PREFIX + f"\nQuestion: {question.strip()}\nAnswer:"
    enc = tokenizer(prompt, return_tensors="pt").to(device)
    prompt_len = enc["input_ids"].shape[1]

    timing = _TimingCriteria()
    sync_if_cuda()
    gen_start = time.perf_counter()
    gen_out = model_obj.generate(
        **enc, max_new_tokens=GSM8K_MAX_NEW_TOKENS, do_sample=False,
        pad_token_id=tokenizer.pad_token_id,
        stopping_criteria=StoppingCriteriaList([timing]),
        output_scores=True, return_dict_in_generate=True,
    )
    sync_if_cuda()
    gen_end = time.perf_counter()
    total_latency_sec = gen_end - gen_start

    gen_ids = gen_out.sequences
    step_scores = gen_out.scores

    gen_text = tokenizer.decode(gen_ids[0][prompt_len:], skip_special_tokens=True)
    gen_text = gen_text.split("Question:")[0]

    n_generated = len(timing.token_times)
    if n_generated > 0:
        ttft_sec = timing.token_times[0] - gen_start
    else:
        ttft_sec = total_latency_sec
    tbt_sec = (total_latency_sec - ttft_sec) / max(n_generated - 1, 1)

    total_tokens = prompt_len + n_generated

    # Perplexity of the model's own predicted final number -- everything
    # below runs after gen_end, so none of it can affect any timing number.
    # step_scores[i] are the logits that chose the i-th generated token
    # (already computed by generate() above -- no extra forward pass).
    answer_token_ids = gen_ids[0][prompt_len: prompt_len + len(step_scores)].tolist()
    hash_streak = 0
    in_answer_span = False
    nll_sum = 0.0
    scored = 0
    for i, tok_id in enumerate(answer_token_ids):
        tok_text = tokenizer.decode([tok_id])
        if in_answer_span:
            if "#" in tok_text:
                break
            log_probs = torch.log_softmax(step_scores[i][0].float(), dim=-1)
            nll_sum += -log_probs[tok_id].item()
            scored += 1
        else:
            for ch in tok_text:
                hash_streak = hash_streak + 1 if ch == "#" else 0
            if hash_streak >= 4:
                in_answer_span = True
    perplexity = math.exp(min(nll_sum / scored, 50.0)) if scored > 0 else None

    # Untimed pass over the full generated sequence, purely to measure
    # memory -- real outlier-hook tracker for the quantized model. Does not
    # affect the timed generate() call above.
    if tracker is not None:
        reset_outlier_tracker(tracker)
        _tracker_enabled[0] = True
        try:
            model_obj(input_ids=gen_ids, use_cache=False, return_dict=True)
        finally:
            _tracker_enabled[0] = False
        peak_bytes = measure_bytes_from_tracker(tracker, bits_per_element)
    else:
        cache_outputs = model_obj(input_ids=gen_ids, use_cache=True, return_dict=True)
        pkv = cache_outputs.past_key_values
        legacy = pkv.to_legacy_cache() if hasattr(pkv, "to_legacy_cache") else pkv
        peak_bytes = sum(t.numel() * t.element_size() for layer_kv in legacy for t in layer_kv)

    return {
        "gen_text": gen_text, "ttft_sec": ttft_sec, "tbt_sec": tbt_sec,
        "total_latency_sec": total_latency_sec, "total_tokens": total_tokens,
        "peak_memory_bytes": peak_bytes, "perplexity": perplexity,
    }


In [ ]:
# Block - GSM8K driver: run every question through generate_gsm8k_kvquant
# (accuracy + TTFT/TBT/latency + memory + perplexity, all from the SAME
# real generation call -- perplexity is measured live on the model's own
# generated answer, no separate teacher-forced pass) -- against the
# 2-bit quantized model.


def evaluate_gsm8k_kvquant(qa_pairs, model_obj, bits_per_element, method_label, tracker=None):
    clear_memory()
    correct = 0
    total = 0
    ttft_values, tbt_values, latency_values, peak_mem_values, ppl_values = [], [], [], [], []
    per_question_records = []

    N_PREVIEW_QUESTIONS = 5
    for q_idx, qa in enumerate(tqdm(qa_pairs, desc=f"GSM8K | {method_label}")):
        result = generate_gsm8k_kvquant(model_obj, qa["question"], bits_per_element, tracker)
        pred = _extract_final_number(result["gen_text"])
        is_correct = pred is not None and abs(pred - qa["gold"]) < 1e-4
        correct += int(is_correct)
        total += 1

        if q_idx < N_PREVIEW_QUESTIONS:
            print(f"\n--- GSM8K | {method_label} | question {q_idx} preview ---")
            print(f"Question:    {qa['question']}")
            print(f"Generated:   {result['gen_text'].strip()}")
            print(f"Gold answer: {qa['gold']} | Predicted: {pred} | Correct: {is_correct}")

        ttft_values.append(result["ttft_sec"])
        tbt_values.append(result["tbt_sec"])
        latency_values.append(result["total_latency_sec"])
        peak_mem_values.append(result["peak_memory_bytes"])

        ppl = result["perplexity"]
        if ppl is not None:
            ppl_values.append(ppl)

        per_question_records.append({
            "question_index": q_idx,
            "prompt": qa["question"],
            "gold_answer": qa["gold"],
            "predicted_answer": pred,
            "correct": int(is_correct),
            "perplexity": ppl,
            "ttft_sec": result["ttft_sec"],
            "tbt_sec": result["tbt_sec"],
            "total_latency_sec": result["total_latency_sec"],
            "peak_memory_gb": result["peak_memory_bytes"] / 1024**3,
        })

    accuracy = correct / max(total, 1)
    avg_ppl = sum(ppl_values) / len(ppl_values) if ppl_values else float("nan")

    os.makedirs("/content/drive/MyDrive/KVQuant_v3_Results", exist_ok=True)
    _per_prompt_path = f"/content/drive/MyDrive/KVQuant_v3_Results/{method_label}_gsm8k_per_prompt.csv"
    pd.DataFrame(per_question_records).to_csv(_per_prompt_path, index=False)
    print(f"Saved {len(per_question_records)} per-question GSM8K rows to {_per_prompt_path}")

    return {
        "dataset": "GSM8K",
        "method": method_label,
        "perplexity": avg_ppl,
        "accuracy": accuracy,
        "ttft_sec": sum(ttft_values) / len(ttft_values) if ttft_values else float("nan"),
        "tbt_sec": sum(tbt_values) / len(tbt_values) if tbt_values else float("nan"),
        "avg_total_latency_sec": sum(latency_values) / len(latency_values) if latency_values else float("nan"),
        "peak_memory_gb": max(peak_mem_values) / 1024**3 if peak_mem_values else 0.0,
        "average_memory_gb": (sum(peak_mem_values) / len(peak_mem_values) / 1024**3) if peak_mem_values else 0.0,
    }


gsm8k_results = [
    evaluate_gsm8k_kvquant(gsm8k_qa_pairs, model_q, ABITS, METHOD_NAME, tracker=outlier_tracker),
]
gsm8k_results_df = pd.DataFrame(gsm8k_results)
display(gsm8k_results_df)

## ARC-Challenge

Likelihood-based multiple-choice scoring (no generation): every answer
choice is scored (raw and character-length-normalized), perplexity from
the gold choice, TTFT/TBT/latency/memory aggregated across all choices.

In [ ]:
# Block - ARC-Challenge loading: canonical test split, validity filtering,
# then random sampling. Invalid answer-key rows are removed before selecting
# up to 1,024 valid questions with seed 42.


def load_arc_challenge_items():
    testdata = robust_call(
        load_dataset, "allenai/ai2_arc", "ARC-Challenge", split="test",
        desc="ARC-Challenge test load", on_retry=lambda: clear_hf_dataset_cache("ai2_arc"),
    )

    valid_items = []
    for row in testdata:
        labels = row["choices"]["label"]
        texts = row["choices"]["text"]
        answer_key = row["answerKey"]
        if answer_key not in labels:
            continue
        valid_items.append({
            "question": row["question"],
            "choices": list(zip(labels, texts)),
            "gold_label": answer_key,
        })

    selected_items, selected_indices = seeded_subset(
        valid_items,
        QA_EVAL_SAMPLES,
        SHARED_SEED,
    )
    print(
        f"ARC-Challenge: {len(valid_items)} valid questions available; "
        f"selected {len(selected_items)} random questions "
        f"(requested {QA_EVAL_SAMPLES}, seed={SHARED_SEED})"
    )
    print("ARC-Challenge selected valid-item indices (first 20):", selected_indices[:20])
    return selected_items


arc_items = load_arc_challenge_items()


In [ ]:
# Block - ARC-Challenge driver: scores every answer choice for each
# question via the shared score_mc_question_kvquant (Helper Functions),
# then reports character-length normalized MC accuracy. Aggregation
# mirrors GSM8K: perplexity = MEAN of per-question perplexities (from
# each question's gold choice), TTFT/TBT/latency = means over questions,
# peak_memory_gb = max over questions, average_memory_gb = mean over
# questions.


def evaluate_arc_kvquant(items, model_obj, bits_per_element, method_label, tracker=None):
    clear_memory()
    correct = 0
    total = 0
    ttft_values, tbt_values, latency_values, peak_mem_values, ppl_values = [], [], [], [], []
    per_item_records = []

    N_PREVIEW_ITEMS = 5
    for idx, item in enumerate(tqdm(items, desc=f"ARC-Challenge | {method_label}")):
        prompt = f"Question: {item['question']}\nAnswer:"
        choice_texts = [text for _, text in item["choices"]]
        gold_index = next(i for i, (label, _) in enumerate(item["choices"]) if label == item["gold_label"])

        result = score_mc_question_kvquant(model_obj, prompt, choice_texts, gold_index, bits_per_element, tracker)

        correct += result["normalized_correct"]
        total += 1

        predicted_label = item["choices"][result["normalized_prediction"]][0]
        if idx < N_PREVIEW_ITEMS:
            print(f"\n--- ARC-Challenge | {method_label} | item {idx} preview ---")
            print(f"Question:   {item['question']}")
            print(f"Choices:    {item['choices']}")
            print(f"Gold label: {item['gold_label']} | Predicted: {predicted_label} | Correct: {bool(result['normalized_correct'])}")

        ttft_values.append(result["ttft_sec"])
        tbt_values.append(result["tbt_sec"])
        latency_values.append(result["total_latency_sec"])
        peak_mem_values.append(result["peak_memory_bytes"])
        ppl_values.append(result["perplexity"])

        per_item_records.append({
            "item_index": idx,
            "prompt": item["question"],
            "choices": item["choices"],
            "gold_label": item["gold_label"],
            "predicted_label": predicted_label,
            "correct": result["normalized_correct"],
            "correct_raw": result["raw_correct"],
            "perplexity": result["perplexity"],
            "ttft_sec": result["ttft_sec"],
            "tbt_sec": result["tbt_sec"],
            "total_latency_sec": result["total_latency_sec"],
            "peak_memory_gb": result["peak_memory_bytes"] / 1024**3,
        })

    accuracy = correct / max(total, 1)
    avg_ppl = sum(ppl_values) / len(ppl_values) if ppl_values else float("nan")

    os.makedirs("/content/drive/MyDrive/KVQuant_v3_Results", exist_ok=True)
    _per_prompt_path = f"/content/drive/MyDrive/KVQuant_v3_Results/{method_label}_arc_challenge_per_prompt.csv"
    pd.DataFrame(per_item_records).to_csv(_per_prompt_path, index=False)
    print(f"Saved {len(per_item_records)} per-item ARC-Challenge rows to {_per_prompt_path}")

    return {
        "dataset": "ARC-Challenge",
        "method": method_label,
        "perplexity": avg_ppl,
        "accuracy": accuracy,
        "ttft_sec": sum(ttft_values) / len(ttft_values) if ttft_values else float("nan"),
        "tbt_sec": sum(tbt_values) / len(tbt_values) if tbt_values else float("nan"),
        "avg_total_latency_sec": sum(latency_values) / len(latency_values) if latency_values else float("nan"),
        "peak_memory_gb": max(peak_mem_values) / 1024**3 if peak_mem_values else 0.0,
        "average_memory_gb": (sum(peak_mem_values) / len(peak_mem_values) / 1024**3) if peak_mem_values else 0.0,
    }


arc_results = [
    evaluate_arc_kvquant(arc_items, model_q, ABITS, METHOD_NAME, tracker=outlier_tracker),
]
arc_results_df = pd.DataFrame(arc_results)
display(arc_results_df)

## HellaSwag

Likelihood-based multiple-choice scoring (no generation), same
methodology as ARC-Challenge: every answer choice is scored (raw and
character-length-normalized), perplexity from the gold ending,
TTFT/TBT/latency/memory aggregated across all choices.

In [ ]:
# Block - HellaSwag loading: canonical validation split, preprocessing,
# then random sampling. HellaSwag's test labels are private, so validation is
# the standard evaluation split. Up to 1,024 processed examples are selected
# with seed 42.


def _hellaswag_preprocess(text):
    """Match lm-evaluation-harness HellaSwag preprocessing."""
    text = str(text).strip()
    text = text.replace(" [title]", ". ")
    text = re.sub(r"\[.*?\]", "", text)
    text = text.replace("  ", " ")
    return text


def load_hellaswag_items():
    dataset = robust_call(
        load_dataset, "Rowan/hellaswag", split="validation",
        desc="HellaSwag validation load", on_retry=lambda: clear_hf_dataset_cache("hellaswag"),
    )

    processed_items = []
    for row in dataset:
        context = str(row["ctx_a"]) + " " + str(row["ctx_b"]).capitalize()
        prompt = _hellaswag_preprocess(str(row["activity_label"]) + ": " + context)
        choices = [_hellaswag_preprocess(choice) for choice in row["endings"]]
        processed_items.append({
            "prompt": prompt,
            "choices": choices,
            "gold_index": int(row["label"]),
        })

    selected_items, selected_indices = seeded_subset(
        processed_items,
        QA_EVAL_SAMPLES,
        SHARED_SEED,
    )
    print(
        f"HellaSwag: {len(processed_items)} validation examples available; "
        f"selected {len(selected_items)} random examples "
        f"(requested {QA_EVAL_SAMPLES}, seed={SHARED_SEED})"
    )
    print("HellaSwag selected example indices (first 20):", selected_indices[:20])
    return selected_items


hellaswag_items = load_hellaswag_items()


In [ ]:
# Block - HellaSwag driver: scores every answer choice for each example
# via the shared score_mc_question_kvquant (Helper Functions) -- the same
# machinery ARC-Challenge uses, since both are likelihood-based
# multiple-choice datasets. Aggregation is identical to ARC-Challenge:
# normalized accuracy, perplexity = MEAN of per-question perplexities
# (from each example's gold ending), TTFT/TBT/latency = means over
# examples, peak_memory_gb = max over examples, average_memory_gb =
# mean over examples.


def evaluate_hellaswag_kvquant(items, model_obj, bits_per_element, method_label, tracker=None):
    clear_memory()
    correct = 0
    total = 0
    ttft_values, tbt_values, latency_values, peak_mem_values, ppl_values = [], [], [], [], []
    per_item_records = []

    N_PREVIEW_ITEMS = 5
    for idx, item in enumerate(tqdm(items, desc=f"HellaSwag | {method_label}")):
        result = score_mc_question_kvquant(
            model_obj, item["prompt"], item["choices"], item["gold_index"], bits_per_element, tracker,
        )

        correct += result["normalized_correct"]
        total += 1

        if idx < N_PREVIEW_ITEMS:
            print(f"\n--- HellaSwag | {method_label} | item {idx} preview ---")
            print(f"Prompt:     {item['prompt']}")
            print(f"Choices:    {item['choices']}")
            print(f"Gold index: {item['gold_index']} | Predicted: {result['normalized_prediction']} | Correct: {bool(result['normalized_correct'])}")

        ttft_values.append(result["ttft_sec"])
        tbt_values.append(result["tbt_sec"])
        latency_values.append(result["total_latency_sec"])
        peak_mem_values.append(result["peak_memory_bytes"])
        ppl_values.append(result["perplexity"])

        per_item_records.append({
            "item_index": idx,
            "prompt": item["prompt"],
            "choices": item["choices"],
            "gold_index": item["gold_index"],
            "predicted_index": result["normalized_prediction"],
            "correct": result["normalized_correct"],
            "correct_raw": result["raw_correct"],
            "perplexity": result["perplexity"],
            "ttft_sec": result["ttft_sec"],
            "tbt_sec": result["tbt_sec"],
            "total_latency_sec": result["total_latency_sec"],
            "peak_memory_gb": result["peak_memory_bytes"] / 1024**3,
        })

    accuracy = correct / max(total, 1)
    avg_ppl = sum(ppl_values) / len(ppl_values) if ppl_values else float("nan")

    os.makedirs("/content/drive/MyDrive/KVQuant_v3_Results", exist_ok=True)
    _per_prompt_path = f"/content/drive/MyDrive/KVQuant_v3_Results/{method_label}_hellaswag_per_prompt.csv"
    pd.DataFrame(per_item_records).to_csv(_per_prompt_path, index=False)
    print(f"Saved {len(per_item_records)} per-item HellaSwag rows to {_per_prompt_path}")

    return {
        "dataset": "HellaSwag",
        "method": method_label,
        "perplexity": avg_ppl,
        "accuracy": accuracy,
        "ttft_sec": sum(ttft_values) / len(ttft_values) if ttft_values else float("nan"),
        "tbt_sec": sum(tbt_values) / len(tbt_values) if tbt_values else float("nan"),
        "avg_total_latency_sec": sum(latency_values) / len(latency_values) if latency_values else float("nan"),
        "peak_memory_gb": max(peak_mem_values) / 1024**3 if peak_mem_values else 0.0,
        "average_memory_gb": (sum(peak_mem_values) / len(peak_mem_values) / 1024**3) if peak_mem_values else 0.0,
    }


hellaswag_results = [
    evaluate_hellaswag_kvquant(hellaswag_items, model_q, ABITS, METHOD_NAME, tracker=outlier_tracker),
]
hellaswag_results_df = pd.DataFrame(hellaswag_results)
display(hellaswag_results_df)

## Save Results

In [ ]:
# Block - Combine WikiText-103 / GSM8K / ARC-Challenge / HellaSwag results into
# one table with ONLY the specified metrics, then save to CSV. This notebook only ever
# produces "kvquant_2bit" rows -- 3-bit, 4-bit, and the full-precision
# baseline are saved by their own separate notebooks, so there is no
# cross-notebook filename collision risk.
#
# Robust to partial runs: only concatenates whichever of
# lm_results_df / gsm8k_results_df / arc_results_df / hellaswag_results_df
# actually exist in this session, so you can run just a subset of
# datasets' cells without this cell crashing on a NameError for a
# dataframe you never created.

_result_df_names = ["lm_results_df", "gsm8k_results_df", "arc_results_df", "hellaswag_results_df"]
_available_dfs = []
for _name in _result_df_names:
    if _name in globals():
        _available_dfs.append(globals()[_name])
    else:
        print(f"Note: {_name} not found in this session -- skipping (its dataset's cells were not run).")

results_df = pd.concat(_available_dfs, ignore_index=True)
results_df = results_df[[
    "dataset", "method", "perplexity", "accuracy",
    "ttft_sec", "tbt_sec", "avg_total_latency_sec",
    "peak_memory_gb", "average_memory_gb",
]]
display(results_df)

os.makedirs("/content/drive/MyDrive/KVQuant_v3_Results", exist_ok=True)

_path = "/content/drive/MyDrive/KVQuant_v3_Results/kvquant_abits2_results.csv"
results_df.to_csv(_path, index=False)
print(f"Saved to {_path}")